In [9]:
import numpy as np
from scipy.optimize import minimize
import optimize
from pathlib import Path
import numpy as np
import pandas as pd

In [8]:
# Little random example with 3 assets and cov marrix
mu = np.array([0.08, 0.10, 0.12])
cov = np.array([
    [0.18, 0.03, 0.04],
    [0.03, 0.12, 0.02],
    [0.04, 0.02, 0.10],
])

sr_solution = optimize.maximize_sharpe_ratio(mu, cov, risk_free=0.02)
gm_solution = optimize.maximize_geometric_mean(mu, cov)

np.set_printoptions(suppress=False, precision=6)

print("Sharpe weights:", np.round(sr_solution["weights"], 6))
print("Optimal Sharpe:", np.round(sr_solution["optimal_sharpe"], 6))
print("GM weights:", np.round(gm_solution["weights"], 6))
print("Optimal GM:", np.round(gm_solution["optimal_gm"], 6))


Sharpe weights: [0.03751  0.352587 0.609903]
Optimal Sharpe: 0.362629
GM weights: [0.       0.313963 0.686037]
Optimal GM: 0.083822


In [ ]:
# ETF analysis using a recent lookback window

etf_input_names = {
    "Historical Prices - Vanguard FT": ("Date", "NAV (USD)"),
    "iShares-MSCI-World-Small-Cap-UC": ("As Of", "NAV per Share"),
}
lookback_days = 252 * 3  # consider only last 3 years

def load_etf_returns(path="data_investing/etf_returns.xlsx"):
    """Read ETF sheets and return aligned daily returns."""
    prices = {}

    for sheet, (date_col, price_col) in etf_input_names.items():
        df = pd.read_excel(path, sheet_name=sheet, usecols=[date_col, price_col])
        dates = pd.to_datetime(df[date_col], errors="coerce")
        numeric_dates = pd.to_numeric(df[date_col], errors="coerce")
        dates = dates.mask(
            numeric_dates.notna(),
            pd.to_datetime(numeric_dates, unit="D", origin="1899-12-30"),
        )
        values = pd.to_numeric(
            df[price_col].astype(str).str.replace(r"[$,]", "", regex=True),
            errors="coerce",
        )
        prices[sheet] = pd.Series(values.to_numpy(), index=dates).dropna()

    return pd.DataFrame(prices).sort_index().dropna().pct_change().dropna()


all_returns = load_etf_returns()
returns = all_returns.tail(lookback_days)

daily_mu = returns.mean()
daily_cov = returns.cov()
annual_mu = daily_mu * 252
annual_cov = daily_cov * 252
annual_vol = returns.std() * np.sqrt(252)
annual_return = (1 + returns).prod() ** (252 / len(returns)) - 1

stats = pd.DataFrame(
    {
        "historical_return": annual_return,
        "arithmetic_return": annual_mu,
        "volatility": annual_vol,
    }
)
print("\nAnnualized ETF statistics:")
print(stats.T.round(4))

mu, cov = daily_mu.to_numpy(), daily_cov.to_numpy()
sharpe = optimize.maximize_sharpe_ratio(mu, cov)
gm = optimize.maximize_geometric_mean(mu, cov)

weights = pd.DataFrame(
    {
        "ETF": returns.columns,
        "Sharpe_weight": np.round(sharpe["weights"], 6),
        "GM_weight": np.round(gm["weights"], 6),
    }
).set_index("ETF")
print("\nPortfolio weights:")
print(weights.round(4))
print("\nOptimal daily Sharpe:", round(sharpe["optimal_sharpe"], 6))
print("Approx. annualized Sharpe:", round(sharpe["optimal_sharpe"] * np.sqrt(252), 6))
print("Optimal daily GM:", round(gm["optimal_gm"], 6))
print("Approx. annualized GM:", round((1 + gm["optimal_gm"]) ** 252 - 1, 6))



Annualized ETF statistics:
                   Historical Prices - Vanguard FT  \
historical_return                           0.2104   
arithmetic_return                           0.1980   
volatility                                  0.1178   

                   iShares-MSCI-World-Small-Cap-UC  
historical_return                           0.1711  
arithmetic_return                           0.1690  
volatility                                  0.1486  

Portfolio weights:
                                 Sharpe_weight  GM_weight
ETF                                                      
Historical Prices - Vanguard FT            1.0        1.0
iShares-MSCI-World-Small-Cap-UC            0.0        0.0

Optimal daily Sharpe: 0.105852
Approx. annualized Sharpe: 1.680349
Optimal daily GM: 0.000758
Approx. annualized GM: 0.210454
